# Prometheus-1B: Phase 1 - Data Preparation
**다도메인 고품질 데이터 수집, 필터링, 토큰화**

| 도메인 | 데이터셋 | 비율 | 목적 |
|--------|---------|------|------|
| 교육/지식 | FineWeb-Edu (score>=3.0) | 35% | MMLU, ARC-C 향상 |
| 수학 | OpenWebMath | 20% | GSM8K 향상 |
| 코드 | StarCoderData (Python) | 15% | HumanEval 향상 |
| 합성 교과서 | Cosmopedia | 15% | 구조적 지식 (Phi 전략) |
| 과학/추론 | Proof-Pile-2 | 10% | 논리 추론 향상 |
| 리플레이 | SlimPajama | 5% | Catastrophic Forgetting 방지 |

**예상 소요**: 3-5시간 (A100 GPU + 고용량 RAM)  
**출력**: Google Drive에 토큰화된 Arrow 데이터셋  
**체크포인트**: 10,000 시퀀스마다 자동 저장 → 세션 끊겨도 이어서 수집 가능

## 1. 환경 설정

In [ ]:
!pip install -q datasets transformers huggingface_hub

import os
from google.colab import drive, userdata

# Google Drive 마운트
drive.mount('/content/drive')

# HuggingFace 인증 (Colab Secrets에서 HF_TOKEN 설정 필요)
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

# 출력 디렉토리
OUTPUT_DIR = "/content/drive/MyDrive/Prometheus-1B-Data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print("Environment ready!")

In [ ]:
# Colab 세션 끊김 방지 (백그라운드 스레드)
import threading
import time
import IPython

def keep_alive():
    while True:
        time.sleep(120)  # 2분마다
        IPython.display.clear_output(wait=True)
        
keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("Keep-alive thread started (2min interval)")

## 2. 설정값 정의
토큰 수 목표와 도메인 비율을 설정합니다. 전체 ~3.5B 토큰을 목표로 합니다.

In [ ]:
from transformers import AutoTokenizer

# Llama 3.2 토크나이저 로드
# ⚠️ 사전 필수: https://huggingface.co/meta-llama/Llama-3.2-1B 에서 라이선스 동의!
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    token=hf_token,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === 핵심 설정 ===
MAX_SEQ_LENGTH = 4096          # 시퀀스 길이
TOTAL_TOKENS_TARGET = 3_500_000_000  # 3.5B 토큰 목표
CHUNK_SIZE = 10_000  # 10,000 시퀀스마다 중간 저장 (세션 끊김 대비)

# 도메인별 비율 및 토큰 수 할당
# ✅ 커스텀 스크립트 문제 있는 데이터셋 전부 제거, 검증된 것들만 사용
DOMAIN_CONFIG = {
    "fineweb_edu": {
        "ratio": 0.35,
        "tokens": int(TOTAL_TOKENS_TARGET * 0.35),
        "dataset_id": "HuggingFaceFW/fineweb-edu",
        "subset": "sample-100BT",
        "data_dir": None,
        "text_field": "text",
        "filter_fn": lambda x: x.get("score", 0) >= 3.0,
    },
    "openwebmath": {
        "ratio": 0.20,
        "tokens": int(TOTAL_TOKENS_TARGET * 0.20),
        "dataset_id": "open-web-math/open-web-math",
        "subset": None,
        "data_dir": None,
        "text_field": "text",
        "filter_fn": None,
    },
    "stack_python": {
        "ratio": 0.15,
        "tokens": int(TOTAL_TOKENS_TARGET * 0.15),
        "dataset_id": "bigcode/starcoderdata",
        "subset": None,
        "data_dir": "python",
        "text_field": "content",
        "filter_fn": None,
    },
    "cosmopedia": {
        "ratio": 0.15,
        "tokens": int(TOTAL_TOKENS_TARGET * 0.15),
        "dataset_id": "HuggingFaceTB/cosmopedia",
        "subset": "web_samples_v2",
        "data_dir": None,
        "text_field": "text",
        "filter_fn": None,
    },
    "cosmopedia_science": {
        "ratio": 0.10,
        "tokens": int(TOTAL_TOKENS_TARGET * 0.10),
        "dataset_id": "HuggingFaceTB/cosmopedia",
        "subset": "stanford",  # ✅ openstax보다 데이터 많음
        "data_dir": None,
        "text_field": "text",
        "filter_fn": None,
    },
    "cosmopedia_stories": {
        "ratio": 0.05,
        "tokens": int(TOTAL_TOKENS_TARGET * 0.05),  # ~175M
        "dataset_id": "HuggingFaceTB/cosmopedia",  # ✅ SlimPajama 대체
        "subset": "stories",  # 다양한 주제의 스토리 → forgetting 방지
        "data_dir": None,
        "text_field": "text",
        "filter_fn": None,
    },
}

# 요약 출력
print("=" * 60)
print("Prometheus-1B Data Mix Configuration")
print("=" * 60)
for name, cfg in DOMAIN_CONFIG.items():
    print(f"  {name:20s} | {cfg['ratio']*100:5.1f}% | {cfg['tokens']/1e9:.3f}B tokens")
print(f"  {'TOTAL':20s} | 100.0% | {TOTAL_TOKENS_TARGET/1e9:.3f}B tokens")
print(f"  Sequence Length: {MAX_SEQ_LENGTH}")
print(f"  Chunk checkpoint: every {CHUNK_SIZE:,} sequences")
print(f"  Sequences needed: ~{TOTAL_TOKENS_TARGET // MAX_SEQ_LENGTH:,}")

## 3. 데이터 수집 및 토큰화 함수
각 도메인 데이터를 스트리밍으로 로드하고, 토큰화한 뒤, 4096 길이의 청크로 패킹합니다.  
**패킹(Packing)**: 짧은 텍스트 여러 개를 이어 붙여 4096 토큰 시퀀스를 빈틈없이 채우는 방식.  
**Chunk 체크포인트**: 10,000 시퀀스마다 Google Drive에 자동 저장. 세션이 끊겨도 저장된 chunk부터 이어서 수집.

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import numpy as np
from tqdm.auto import tqdm
import gc
import json
import glob as glob_module

def get_existing_chunks(domain_dir):
    """이미 저장된 chunk 목록과 총 시퀀스 수를 반환"""
    if not os.path.exists(domain_dir):
        return [], 0
    
    chunk_dirs = sorted(glob_module.glob(os.path.join(domain_dir, "chunk_*")))
    total_seqs = 0
    valid_chunks = []
    
    for chunk_dir in chunk_dirs:
        try:
            ds = Dataset.load_from_disk(chunk_dir)
            total_seqs += len(ds)
            valid_chunks.append(chunk_dir)
            del ds
        except Exception:
            import shutil
            shutil.rmtree(chunk_dir, ignore_errors=True)
    
    return valid_chunks, total_seqs


def save_chunk(domain_dir, chunk_idx, input_ids_list):
    """chunk를 Drive에 저장"""
    chunk_path = os.path.join(domain_dir, f"chunk_{chunk_idx:04d}")
    ds = Dataset.from_dict({
        "input_ids": input_ids_list,
        "labels": [ids[:] for ids in input_ids_list],
    })
    ds.save_to_disk(chunk_path)
    return chunk_path


def collect_and_tokenize_domain(domain_name, config, tokenizer, max_seq_length, 
                                 chunk_size, domain_dir):
    """
    단일 도메인의 데이터를 스트리밍으로 수집, 토큰화, 패킹.
    chunk_size 시퀀스마다 Drive에 중간 저장.
    이미 저장된 chunk가 있으면 그만큼 건너뛰고 이어서 수집.
    """
    target_tokens = config["tokens"]
    target_sequences = target_tokens // max_seq_length
    
    os.makedirs(domain_dir, exist_ok=True)
    existing_chunks, existing_seqs = get_existing_chunks(domain_dir)
    
    print(f"\n{'='*60}")
    print(f"Processing: {domain_name}")
    print(f"  Target: {target_tokens/1e6:.0f}M tokens ({target_sequences:,} sequences)")
    print(f"  Dataset: {config['dataset_id']}")
    if existing_seqs > 0:
        print(f"  ✅ Resuming: {existing_seqs:,} sequences already saved ({len(existing_chunks)} chunks)")
        print(f"  Remaining: {target_sequences - existing_seqs:,} sequences")
    print(f"{'='*60}")
    
    if existing_seqs >= target_sequences:
        print(f"  ✅ Already complete! ({existing_seqs:,} >= {target_sequences:,})")
        return None
    
    remaining_sequences = target_sequences - existing_seqs
    next_chunk_idx = len(existing_chunks)
    
    # 스트리밍으로 데이터셋 로드
    load_kwargs = {
        "path": config["dataset_id"],
        "split": "train",
        "streaming": True,
        "token": hf_token,
        "trust_remote_code": True,  # ✅ proof-pile-2 등 커스텀 스크립트 지원
    }
    if config.get("subset"):
        load_kwargs["name"] = config["subset"]
    if config.get("data_dir"):
        load_kwargs["data_dir"] = config["data_dir"]
    
    ds_stream = load_dataset(**load_kwargs)
    
    if config["filter_fn"]:
        ds_stream = ds_stream.filter(config["filter_fn"])
    
    # 이미 처리한 문서 수만큼 건너뛰기
    docs_to_skip = 0
    if existing_seqs > 0:
        estimated_tokens_done = existing_seqs * max_seq_length
        print(f"  Skipping ~{estimated_tokens_done/1e6:.0f}M tokens worth of documents...")
        
        tokens_skipped = 0
        text_field = config["text_field"]
        skip_pbar = tqdm(total=int(estimated_tokens_done), desc="  Skipping", unit="tok")
        
        for example in ds_stream:
            text = example.get(text_field, "")
            if not text or len(text.strip()) < 50:
                continue
            est_tokens = len(text) // 4
            tokens_skipped += est_tokens
            docs_to_skip += 1
            skip_pbar.update(est_tokens)
            
            if tokens_skipped >= estimated_tokens_done:
                break
        
        skip_pbar.close()
        print(f"  Skipped {docs_to_skip:,} documents (~{tokens_skipped/1e6:.0f}M tokens)")
    
    # 토큰화 + 패킹
    text_field = config["text_field"]
    token_buffer = []
    chunk_buffer = []
    total_new_seqs = 0
    docs_processed = 0
    
    pbar = tqdm(total=remaining_sequences, desc=f"  Packing {domain_name}", unit="seq")
    
    for example in ds_stream:
        text = example.get(text_field, "")
        if not text or len(text.strip()) < 50:
            continue
        
        tokens = tokenizer.encode(text, add_special_tokens=False)
        token_buffer.extend(tokens)
        token_buffer.append(tokenizer.eos_token_id)
        docs_processed += 1
        
        while len(token_buffer) >= max_seq_length:
            chunk = token_buffer[:max_seq_length]
            chunk_buffer.append(chunk)
            token_buffer = token_buffer[max_seq_length:]
            total_new_seqs += 1
            pbar.update(1)
            
            if len(chunk_buffer) >= chunk_size:
                save_path = save_chunk(domain_dir, next_chunk_idx, chunk_buffer)
                print(f"\n  💾 Chunk {next_chunk_idx} saved: {len(chunk_buffer):,} seqs → {save_path}")
                next_chunk_idx += 1
                chunk_buffer = []
                gc.collect()
            
            if total_new_seqs >= remaining_sequences:
                break
        
        if total_new_seqs >= remaining_sequences:
            break
    
    pbar.close()
    
    if chunk_buffer:
        save_path = save_chunk(domain_dir, next_chunk_idx, chunk_buffer)
        print(f"  💾 Final chunk {next_chunk_idx} saved: {len(chunk_buffer):,} seqs → {save_path}")
        del chunk_buffer
    
    # 완료 마커
    complete_marker = os.path.join(domain_dir, "_COMPLETE")
    total_seqs = existing_seqs + total_new_seqs
    with open(complete_marker, "w") as f:
        f.write(f"{total_seqs}")
    
    actual_tokens = total_seqs * max_seq_length
    print(f"\n  ✅ {domain_name} COMPLETE: {total_seqs:,} sequences ({actual_tokens/1e6:.0f}M tokens)")
    print(f"  New documents processed: {docs_processed:,}")
    
    del token_buffer
    gc.collect()
    
    return None


def load_domain_from_chunks(domain_dir):
    """저장된 chunk들을 합쳐서 하나의 Dataset으로 반환"""
    chunk_dirs = sorted(glob_module.glob(os.path.join(domain_dir, "chunk_*")))
    if not chunk_dirs:
        return None
    
    datasets = []
    for chunk_dir in chunk_dirs:
        try:
            ds = Dataset.load_from_disk(chunk_dir)
            datasets.append(ds)
        except Exception as e:
            print(f"  ⚠️ Failed to load {chunk_dir}: {e}")
    
    if not datasets:
        return None
    
    combined = concatenate_datasets(datasets)
    return combined

## 4. 전체 도메인 데이터 수집 실행
각 도메인을 순차적으로 처리합니다.

> **체크포인트 시스템**:  
> - 10,000 시퀀스(~40M 토큰)마다 Google Drive에 자동 저장  
> - 세션 끊기면 "모두 실행" 다시 누르면 됨 → 저장된 chunk는 자동 스킵  
> - `_COMPLETE` 마커가 있는 도메인은 완전히 건너뜀  
> - 최악의 경우에도 5-10분 분량만 재수집

In [ ]:
domain_datasets = {}

for domain_name, config in DOMAIN_CONFIG.items():
    domain_dir = os.path.join(OUTPUT_DIR, f"domain_{domain_name}")
    complete_marker = os.path.join(domain_dir, "_COMPLETE")
    
    # _COMPLETE 마커가 있으면 완전히 건너뜀
    if os.path.exists(complete_marker):
        print(f"\n[SKIP] {domain_name} - already complete ✅")
        ds = load_domain_from_chunks(domain_dir)
        if ds is not None:
            domain_datasets[domain_name] = ds
            print(f"  Loaded: {len(ds):,} sequences from chunks")
        continue
    
    # 데이터 수집 (chunk 단위 중간 저장 포함)
    collect_and_tokenize_domain(
        domain_name, config, tokenizer, MAX_SEQ_LENGTH,
        CHUNK_SIZE, domain_dir
    )
    
    # 완료 후 chunk에서 로드
    ds = load_domain_from_chunks(domain_dir)
    if ds is not None:
        domain_datasets[domain_name] = ds
    
    gc.collect()

print(f"\n{'='*60}")
print("All domains collected!")
for name, ds in domain_datasets.items():
    print(f"  {name:20s}: {len(ds):,} sequences")
total_seqs = sum(len(ds) for ds in domain_datasets.values())
print(f"  {'TOTAL':20s}: {total_seqs:,} sequences ({total_seqs * MAX_SEQ_LENGTH / 1e9:.2f}B tokens)")

## 5. 도메인 인터리빙 + 셔플 + Train/Eval 분할
도메인 데이터를 합치고 셔플합니다. 1%를 eval set으로 분리합니다.

In [ ]:
import random

# 모든 도메인 합치기
print("Concatenating all domains...")
combined = concatenate_datasets(list(domain_datasets.values()))
print(f"  Combined: {len(combined):,} sequences")

# 셔플 (seed 고정으로 재현 가능)
print("Shuffling...")
combined = combined.shuffle(seed=42)

# Train/Eval 분할 (1% eval)
split = combined.train_test_split(test_size=0.01, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"\n  Train: {len(train_dataset):,} sequences ({len(train_dataset) * MAX_SEQ_LENGTH / 1e9:.2f}B tokens)")
print(f"  Eval:  {len(eval_dataset):,} sequences ({len(eval_dataset) * MAX_SEQ_LENGTH / 1e6:.0f}M tokens)")

# Google Drive에 최종 저장
train_path = os.path.join(OUTPUT_DIR, "train")
eval_path = os.path.join(OUTPUT_DIR, "eval")

print(f"\nSaving train dataset to {train_path}...")
train_dataset.save_to_disk(train_path)

print(f"Saving eval dataset to {eval_path}...")
eval_dataset.save_to_disk(eval_path)

print("\nData preparation complete!")
print(f"  Train path: {train_path}")
print(f"  Eval path:  {eval_path}")

## 6. 데이터 검증
저장된 데이터가 올바른지 확인합니다.

In [ ]:
from datasets import load_from_disk

# 저장된 데이터 다시 로드하여 검증
train_check = load_from_disk(train_path)
eval_check = load_from_disk(eval_path)

print("=== Data Verification ===")
print(f"Train dataset: {len(train_check):,} sequences")
print(f"Eval dataset:  {len(eval_check):,} sequences")
print(f"Features: {train_check.features}")

# 샘플 디코딩 확인
sample = train_check[0]["input_ids"]
print(f"\nSample sequence length: {len(sample)}")
print(f"First 200 tokens decoded:")
print(tokenizer.decode(sample[:200]))
print("...")

# 통계
total_train_tokens = len(train_check) * MAX_SEQ_LENGTH
total_eval_tokens = len(eval_check) * MAX_SEQ_LENGTH
print(f"\n=== Final Statistics ===")
print(f"Total train tokens: {total_train_tokens/1e9:.2f}B")
print(f"Total eval tokens:  {total_eval_tokens/1e6:.0f}M")
print(f"Expected training steps (batch=32): {total_train_tokens // (32 * MAX_SEQ_LENGTH):,}")
print(f"\nData is ready for Phase 2 (CPT Training)!")